# CanvasXpress in VS Code — JavaScript (Deno kernel)

This notebook uses the **native** `new CanvasXpress({...})` call — the same constructor
arguments you would pass in a browser.

A notebook kernel has no DOM, so the constructor cannot draw the chart itself. The
`canvasxpress.mjs` shim gives the object a Jupyter display representation instead, and the
**CanvasXpress Notebook Renderer** extension draws it in the cell output.

> **Requirements:** the Deno kernel and the CanvasXpress Notebook Renderer extension.
>
> Install the kernel once, then pick **Deno** in the kernel picker:
> ```
> deno jupyter --install
> ```

## Import

The shim ships with the extension repo at `kernel-js/canvasxpress.mjs`. Adjust the path if you
keep the notebook elsewhere.

In [ ]:
import { CanvasXpress, fromRows } from "../kernel-js/canvasxpress.mjs";

## 1. Bar chart

Leave the chart as the cell's last expression and Deno displays it.

In [ ]:
new CanvasXpress({
  renderTo: "bar1",
  data: {
    y: {
      vars: ["Revenue"],
      smps: ["Q1", "Q2", "Q3", "Q4"],
      data: [[10, 14, 9, 17]],
    },
  },
  config: {
    graphType: "Bar",
    title: "Quarterly Revenue",
    xAxis: ["Revenue"],
    smpTitle: "Quarter",
    showLegend: false,
  },
});

## 2. Scatter plot

For `Scatter2D` the `smps` are the axis dimensions and the `vars` are the point ids.

In [ ]:
new CanvasXpress({
  renderTo: "scatter1",
  data: {
    y: {
      smps: ["Weight", "MPG"],
      vars: ["c1", "c2", "c3", "c4", "c5", "c6", "c7", "c8"],
      data: [
        [3.2, 24], [2.8, 30], [4.1, 18], [3.6, 21],
        [2.4, 33], [3.9, 19], [2.1, 36], [4.5, 15],
      ],
    },
  },
  config: {
    graphType: "Scatter2D",
    title: "Weight vs. MPG",
    xAxis: ["Weight"],
    yAxis: ["MPG"],
  },
});

## 3. Heatmap from a row-oriented table

JavaScript has no `DataFrame`, so `fromRows()` covers the same common case the R and Python
packages handle for a `data.frame` / `DataFrame`: **row keys become `vars`, column keys become
`smps`**.

In [ ]:
const expr = {
  "Gene A": { S1: 1.2, S2: -0.4, S3: 0.8, S4: 1.9, S5: -1.1, S6: 0.3 },
  "Gene B": { S1: -0.9, S2: 1.4, S3: 0.2, S4: -0.7, S5: 1.8, S6: -0.5 },
  "Gene C": { S1: 0.5, S2: 0.6, S3: -1.3, S4: 1.1, S5: -0.2, S6: 1.6 },
  "Gene D": { S1: 1.7, S2: -1.2, S3: 0.9, S4: -0.6, S5: 0.4, S6: -1.4 },
};

fromRows(expr, {
  graphType: "Heatmap",
  title: "Expression Heatmap",
  samplesClustered: true,
  variablesClustered: true,
});

## 4. Displaying explicitly

When the chart is not the last expression — inside a loop, say — hand it to
`Deno.jupyter.display` yourself.

In a **Node** kernel (`tslab`, `ijavascript`) there is no last-expression hook at all; use that
kernel's display helper with `chart.toMimeBundle()`:

```js
$$.mime(chart.toMimeBundle());
```

In [ ]:
const quarters = ["Q1", "Q2", "Q3", "Q4"];

for (const [name, values] of [["Revenue", [10, 14, 9, 17]], ["Cost", [7, 8, 8, 11]]]) {
  const chart = new CanvasXpress({
    renderTo: `series_${name}`,
    data: { y: { vars: [name], smps: quarters, data: [values] } },
    config: { graphType: "Line", title: name, showLegend: false },
  });
  await Deno.jupyter.display(chart);
}